In [ ]:
import os
import json
import sys
from tempfile import mkdtemp

from imblearn.over_sampling import ADASYN, SMOTE

# make lib/ importable from the notebooks/ directory
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, Normalizer
from lib import label_images

DATA_DIR = "../data"
LABELS_CSV = DATA_DIR + "/labels.csv"
CACHE_FILE = DATA_DIR + "/split_cache.npz"
IMG_SIZE = (64, 64)
CROP_FRAC = 0.80  # keep central 80% of each axis → drops ~10% border on each side

if not os.path.exists(LABELS_CSV):
    label_images.main()

df = pd.read_csv(LABELS_CSV)

In [ ]:

def load_dicom(path, size=IMG_SIZE, crop_frac=CROP_FRAC):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    # center crop — removes border text/watermarks
    h, w = arr.shape
    ch, cw = int(h * crop_frac), int(w * crop_frac)
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    arr = arr[y0:y0 + ch, x0:x0 + cw]
    # resize and flatten
    img = Image.fromarray((arr * 255).astype(np.uint8)).resize(size)
    return np.array(img).flatten()


# NOTE: delete data/split_cache.npz whenever CROP_FRAC changes — the cache
# stores flattened pixels and will be stale if the crop parameters differ.
if os.path.exists(CACHE_FILE):
    print(f"Loading split from cache: {CACHE_FILE}")
    print(f"  (cached with CROP_FRAC={CROP_FRAC} — delete cache if this changed)")
    cache = np.load(CACHE_FILE)
    X_train, X_test, y_train, y_test = cache["X_train"], cache["X_test"], cache["y_train"], cache["y_test"]
else:
    df = pd.read_csv(LABELS_CSV)
    print(f"Total images: {len(df)}  |  sick: {(df.label == 1).sum()}  |  not_sick: {(df.label == 0).sum()}")
    print(f"CROP_FRAC={CROP_FRAC}  IMG_SIZE={IMG_SIZE}")

    print("Loading images...")
    X = np.array([load_dicom(p) for p in df["dicom_path"]])
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    np.savez_compressed(CACHE_FILE, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
    print(f"Split cached to: {CACHE_FILE}")

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

model = models.Sequential([
    Input(shape=(64, 64, 1)),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),  # binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# Reshape flat vectors → (N, 64, 64, 1) and normalise to [0, 1]
X_tr = X_train.reshape(-1, 64, 64, 1).astype('float32') / 255.0
X_te = X_test.reshape(-1, 64, 64, 1).astype('float32') / 255.0

history = model.fit(
    X_tr, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
)

# Evaluate on held-out test set
y_prob = model.predict(X_te, batch_size=64).squeeze()
y_pred_cnn = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred_cnn, target_names=['not_sick', 'sick']))

In [ ]:
from matplotlib import pyplot as plt

cm = confusion_matrix(y_test, y_pred_cnn)
disp = ConfusionMatrixDisplay(cm, display_labels=['not_sick', 'sick'])
disp.plot(cmap='Blues')
plt.title('CNN — Confusion Matrix')
plt.tight_layout()
plt.show()

# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Loss')
ax1.legend()
ax2.plot(history.history['accuracy'], label='train')
ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()